# RF-DETR: ONNX Export, Optimization & AKS Private Deployment
### Complete End-to-End Pipeline: PyTorch -> ONNX -> FastAPI -> Docker -> ACR -> AKS (Internal Load Balancer - No Public IP)

This notebook provides an automated, production-ready pipeline for converting a trained **RF-DETR** object detection model to ONNX and deploying it as a secure, private microservice on **Azure Kubernetes Service (AKS)**.

> **Security & Zero Public Exposure**:
> - Deployed behind an **Azure Internal Load Balancer** (`service.beta.kubernetes.io/azure-load-balancer-internal: "true"`).
> - **NO Public IP** is allocated; the service receives a private VNet IP (`10.x.x.x`).
> - Testing from outside the VNet uses authenticated TLS tunnels via `kubectl port-forward`.

---

### Pipeline Architecture

```
[ RF-DETR (.pth) ] 
       │
       ▼ (Phase 1: Conversion & Validation)
[ ONNX Model (.onnx) ] ──> ONNX Runtime Graph Checker & Latency Benchmark
       │
       ▼ (Phase 2: Microservice Packaging)
[ deployment_artifacts/ ]
   ├── app.py              (FastAPI Inference Microservice)
   ├── Dockerfile          (Production Container Specification)
   ├── requirements.txt    (Lightweight runtime dependencies)
   └── k8s/                (Kubernetes Deployment, Internal Service, HPA)
       │
       ▼ (Phase 3: Azure Cloud Deployment)
[ Azure Container Registry ] ──(az acr build)──> Cloud Image Build
       │
       ▼ (Phase 4: AKS Orchestration)
[ Azure Kubernetes Service ] ──(kubectl apply)──> Internal LoadBalancer (Private VNet IP)
       │
       ▼ (Phase 5: Secure Verification)
[ Private VNet / Port-Forward ] ──> Bounding Box & Label Visualizations
```

---

### Prerequisites
1. **Python Environment**: `torch`, `torchvision`, `onnx`, `onnxruntime`, `pillow`, `requests`.
2. **Azure Account**: Active Azure subscription with permissions to manage Resource Groups, ACR, and AKS.
3. **Azure CLI & Kubectl**: Installed locally or inside your cloud Jupyter environment (`az` and `kubectl`).

In [ ]:
# STEP 0: Install Dependencies
# Run this cell to ensure all required export, serving, and cloud tools are installed.
import sys

%pip install -q "onnx>=1.15.0" "onnxruntime>=1.17.0" "onnxsim>=0.4.33" "fastapi>=0.110.0" "uvicorn[standard]>=0.28.0" "python-multipart>=0.0.9" "pydantic>=2.0.0" "pillow>=10.0.0" "requests>=2.28.0" "matplotlib>=3.7.0"

# Optional: Install rfdetr if not already present
try:
    import rfdetr
    print("✓ rfdetr is already installed.")
except ImportError:
    print("Installing rfdetr with ONNX support...")
    %pip install -q "rfdetr[onnx]"

print("✓ Environment dependencies configured.")

In [ ]:
# CELL 1: Imports & Diagnostic Verifications
import os
import sys
import json
import time
import shutil
import subprocess
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple

import numpy as np
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision.transforms.functional as TF

import onnx
import onnxruntime as ort

print(f"Python Version:      {sys.version.split()[0]}")
print(f"PyTorch Version:     {torch.__version__}")
print(f"ONNX Version:        {onnx.__version__}")
print(f"ONNX Runtime Version:{ort.__version__}")
print(f"CUDA Available:      {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:          {torch.cuda.get_device_name(0)}")

# Check for Azure CLI (az) and kubectl
def check_cli_tool(cmd_name: str) -> bool:
    path = shutil.which(cmd_name)
    if path:
        try:
            ver = subprocess.check_output([cmd_name, "--version"], stderr=subprocess.STDOUT, text=True).splitlines()[0]
            print(f"✓ {cmd_name:<8} found: {ver}")
            return True
        except Exception:
            print(f"✓ {cmd_name:<8} found at {path}")
            return True
    else:
        print(f"⚠ {cmd_name:<8} not found in PATH. (Needed for cloud deployment cells)")
        return False

has_az = check_cli_tool("az")
has_kubectl = check_cli_tool("kubectl")

In [ ]:
# CELL 2: Global Pipeline Configuration

# ----------------- Model & Export Parameters -----------------
MODEL_FAMILY = "RFDETRBase"     # Options: RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRBase, RFDETRLarge
RESOLUTION = 1008               # Input resolution (1008x1008 or 640x640)
CONF_THRESHOLD = 0.25           # Default confidence threshold for inference
IOU_THRESHOLD = 0.50            # NMS IoU threshold

# Target class labels (will be auto-adapted from checkpoint if available)
CLASSES = ["blue_aisle", "blue_bay", "location_tag"]

# Checkpoint candidate search paths
CANDIDATE_CKPTS = [
    "multi_class_train_rfdetr/model/best_model_full_data.pth",
    "multi_class_train_rfdetr/model/best_model_sample_1000.pth",
    "multi_class_train_rfdetr/runs/rf_detr_full_data/checkpoints/best_loss.pth",
    "location_tag_single_class_train_rfdetr/model/best_model_full_data.pth",
    "best_model_full_data.pth",
    "best_model.pth"
]

# Output directories
ARTIFACTS_DIR = Path("deployment_artifacts")
K8S_DIR = ARTIFACTS_DIR / "k8s"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
K8S_DIR.mkdir(parents=True, exist_ok=True)

ONNX_MODEL_PATH = ARTIFACTS_DIR / "rfdetr_model.onnx"

# ----------------- Azure & AKS Parameters -----------------
AZURE_SUBSCRIPTION_ID = ""                    # Optional: leave empty to use default active subscription
AZURE_RESOURCE_GROUP  = "rg-rfdetr-inference"  # Name of Azure Resource Group
AZURE_LOCATION        = "eastus"              # Azure region (e.g. eastus, westeurope, southcentralus)
ACR_NAME              = f"acrrfdetr{int(time.time()) % 1000000:06d}" # ACR names must be alphanumeric lowercase, 5-50 chars
AKS_CLUSTER_NAME      = "aks-rfdetr-cluster"  # Name of AKS cluster
IMAGE_NAME            = "rfdetr-inference"    # Container image repository
IMAGE_TAG             = "v1"                  # Image tag

# Hardware Target for AKS Node Pool:
# - CPU node: 'Standard_D4s_v5' (4 vCPUs, 16 GiB RAM - cost effective, no GPU quota required)
# - GPU node: 'Standard_NC4as_T4_v3' (1 NVIDIA T4 GPU) or 'Standard_NC6s_v3' (1 NVIDIA V100 GPU)
USE_GPU_NODE          = False
NODE_VM_SIZE          = "Standard_NC4as_T4_v3" if USE_GPU_NODE else "Standard_D4s_v5"
NODE_COUNT            = 2                     # Number of worker nodes in default node pool

print("Configuration parameters initialized successfully:")
print(f" - Model:       {MODEL_FAMILY} @ {RESOLUTION}x{RESOLUTION}")
print(f" - Classes:     {CLASSES} ({len(CLASSES)} classes)")
print(f" - Output Dir:  {ARTIFACTS_DIR.resolve()}")
print(f" - ONNX Target: {ONNX_MODEL_PATH.resolve()}")
print(f" - Azure Group: {AZURE_RESOURCE_GROUP} ({AZURE_LOCATION})")
print(f" - ACR Name:    {ACR_NAME}")
print(f" - AKS Cluster: {AKS_CLUSTER_NAME} (VM: {NODE_VM_SIZE}, Nodes: {NODE_COUNT})")

In [ ]:
# CELL 3: Discover and Load RF-DETR Model Checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Find valid checkpoint
active_ckpt_path = None
for cp in CANDIDATE_CKPTS:
    if os.path.exists(cp) and os.path.getsize(cp) > 1024 * 1024:
        active_ckpt_path = cp
        break

print(f"Checkpoint status: {active_ckpt_path if active_ckpt_path else 'No local trained .pth found, will use default weights'}")

# Helper to instantiate model architecture
def get_rfdetr_model(model_name: str, num_classes: int, resolution: int):
    try:
        from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRBase, RFDETRLarge
        from rfdetr.models.lwdetr import LWDETR
        
        cls_map = {
            "rfdetrnano": RFDETRNano,
            "rfdetrsmall": RFDETRSmall,
            "rfdetrmedium": RFDETRMedium,
            "rfdetrbase": RFDETRBase,
            "rfdetrlarge": RFDETRLarge
        }
        ModelClass = cls_map.get(model_name.lower(), RFDETRBase)
        
        # Disable auto-downloads during standalone conversion
        if hasattr(ModelClass, "maybe_download_pretrain_weights"):
            ModelClass.maybe_download_pretrain_weights = lambda self: None
        if hasattr(ModelClass, "load_pretrain_weights"):
            ModelClass.load_pretrain_weights = lambda self: None
            
        wrapper = ModelClass(num_classes=num_classes, resolution=resolution, pretrain_weights=None)
        
        # Extract underlying torch nn.Module
        if hasattr(wrapper, "model") and hasattr(wrapper.model, "model") and isinstance(wrapper.model.model, torch.nn.Module):
            model_core = wrapper.model.model
        elif hasattr(wrapper, "model") and isinstance(wrapper.model, torch.nn.Module):
            model_core = wrapper.model
        else:
            model_core = wrapper
            
        return wrapper, model_core
    except Exception as e:
        print(f"Notice during direct rfdetr import: {e}")
        return None, None

wrapper, model = get_rfdetr_model(MODEL_FAMILY, len(CLASSES), RESOLUTION)

# Load state dict if checkpoint exists
if active_ckpt_path and model is not None:
    print(f"Loading checkpoint weights from: {active_ckpt_path}")
    checkpoint = torch.load(active_ckpt_path, map_location="cpu", weights_only=False)
    state = checkpoint.get("model", checkpoint)
    
    # Inspect checkpoint class heads
    for k in ["class_embed.weight", "model.class_embed.weight", "module.class_embed.weight"]:
        if k in state:
            ckpt_classes = state[k].shape[0]
            if hasattr(model, "class_embed") and model.class_embed.out_features != ckpt_classes:
                print(f"Adapting detection head to checkpoint {ckpt_classes} classes...")
                in_f = model.class_embed.in_features
                model.class_embed = nn.Linear(in_f, ckpt_classes)
            break
            
    filtered = {k.replace("model.", "").replace("module.", ""): v for k, v in state.items()}
    cur_state = model.state_dict()
    matched = {k: v for k, v in filtered.items() if k in cur_state and cur_state[k].shape == v.shape}
    model.load_state_dict(matched, strict=False)
    print(f"✓ Successfully loaded {len(matched)} / {len(cur_state)} tensor layers from checkpoint.")

if model is not None:
    model.eval()
    print("✓ RF-DETR model loaded and set to evaluation mode.")
else:
    print("⚠ RF-DETR library not installed in this kernel. Make sure 'rfdetr' is installed if exporting locally.")

In [ ]:
# CELL 4: Export RF-DETR Model to ONNX

class RFDETRONNXExportWrapper(nn.Module):
    """
    PyTorch Wrapper for seamless ONNX tracing:
    - Accepts standard tensor input: (batch_size, 3, height, width)
    - Returns standardized outputs:
        scores: (batch_size, num_queries, num_classes) [after Sigmoid]
        boxes:  (batch_size, num_queries, 4)           [normalized (cx, cy, w, h)]
    """
    def __init__(self, core_model: nn.Module):
        super().__init__()
        self.model = core_model

    def forward(self, images: torch.Tensor):
        out = self.model(images)
        if isinstance(out, dict):
            pred_logits = out["pred_logits"]
            pred_boxes = out["pred_boxes"]
        elif isinstance(out, (list, tuple)):
            pred_logits, pred_boxes = out[0], out[1]
        else:
            raise ValueError(f"Unexpected output type from model forward: {type(out)}")
            
        scores = torch.sigmoid(pred_logits)
        return scores, pred_boxes

def export_model_to_onnx(model_to_export, wrapper_obj, output_path: Path, resolution: int):
    print(f"Starting ONNX export -> '{output_path}' (Resolution: {resolution}x{resolution})...")
    
    # Method 1: Check if wrapper has native .export() method
    exported_native = False
    if wrapper_obj is not None and hasattr(wrapper_obj, "export"):
        try:
            print("Attempting export via native wrapper.export(format='onnx')...")
            wrapper_obj.export(format="onnx", output_dir=str(output_path.parent))
            # Locate newly created onnx file if differently named
            potential_files = list(output_path.parent.glob("*.onnx"))
            if potential_files:
                for f in potential_files:
                    if f.resolve() != output_path.resolve():
                        shutil.move(str(f), str(output_path))
                exported_native = True
                print("✓ Native export successful!")
        except Exception as e:
            print(f"Native export could not be used ({e}). Falling back to robust torch.onnx.export...")

    # Method 2: Robust PyTorch ONNX Exporter
    if not exported_native:
        if model_to_export is None:
            raise RuntimeError("Model object is not available for export.")
            
        export_wrapper = RFDETRONNXExportWrapper(model_to_export).cpu().eval()
        dummy_input = torch.randn(1, 3, resolution, resolution, dtype=torch.float32)
        
        input_names = ["images"]
        output_names = ["scores", "boxes"]
        dynamic_axes = {
            "images": {0: "batch_size"},
            "scores": {0: "batch_size"},
            "boxes":  {0: "batch_size"}
        }
        
        print("Tracing graph with torch.onnx.export (opset_version=17)...")
        with torch.no_grad():
            torch.onnx.export(
                export_wrapper,
                dummy_input,
                str(output_path),
                export_params=True,
                opset_version=17,
                do_constant_folding=True,
                input_names=input_names,
                output_names=output_names,
                dynamic_axes=dynamic_axes
            )
        print("✓ PyTorch ONNX export completed.")

    # Validate ONNX graph integrity
    print("Validating ONNX model graph structure...")
    onnx_proto = onnx.load(str(output_path))
    onnx.checker.check_model(onnx_proto)
    
    file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"\n=======================================================")
    print(f"✓ ONNX EXPORT SUCCESSFUL: {output_path.name}")
    print(f"   - File Location: {output_path.resolve()}")
    print(f"   - File Size:     {file_size_mb:.2f} MB")
    print(f"   - Graph Inputs:  {[inp.name for inp in onnx_proto.graph.input]}")
    print(f"   - Graph Outputs: {[out.name for out in onnx_proto.graph.output]}")
    print(f"=======================================================\n")

if model is not None:
    export_model_to_onnx(model, wrapper, ONNX_MODEL_PATH, RESOLUTION)
else:
    print(f"Skipping live export in this cell. Ensure '{ONNX_MODEL_PATH}' exists before proceeding to Docker packaging.")

In [ ]:
# CELL 5: ONNX Runtime Inference & Numerical Benchmark

if not ONNX_MODEL_PATH.exists():
    print(f"⚠ ONNX model not found at {ONNX_MODEL_PATH}. Run CELL 4 or place your .onnx model there.")
else:
    print(f"Loading ONNX model for validation from: {ONNX_MODEL_PATH}")
    
    # Configure ONNX Runtime Session
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if torch.cuda.is_available() else ["CPUExecutionProvider"]
    session_options = ort.SessionOptions()
    session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    session_options.intra_op_num_threads = os.cpu_count() or 4
    
    session = ort.InferenceSession(str(ONNX_MODEL_PATH), sess_options=session_options, providers=providers)
    active_provider = session.get_providers()[0]
    print(f"✓ ONNX Runtime Session created using provider: {active_provider}")
    
    # Inspect input and output metadata
    input_info = session.get_inputs()[0]
    print(f"   - Input Name:  '{input_info.name}' | Shape: {input_info.shape} | Type: {input_info.type}")
    for idx, out in enumerate(session.get_outputs()):
        print(f"   - Output [{idx}]: '{out.name}' | Shape: {out.shape} | Type: {out.type}")
        
    # Generate test tensor
    dummy_input = np.random.randn(1, 3, RESOLUTION, RESOLUTION).astype(np.float32)
    
    # Warmup
    for _ in range(3):
        _ = session.run(None, {input_info.name: dummy_input})
        
    # Latency Benchmark
    NUM_ITERS = 20
    latencies = []
    for _ in range(NUM_ITERS):
        t0 = time.perf_counter()
        raw_outs = session.run(None, {input_info.name: dummy_input})
        latencies.append((time.perf_counter() - t0) * 1000)
        
    avg_lat = np.mean(latencies)
    p95_lat = np.percentile(latencies, 95)
    fps = 1000.0 / avg_lat
    
    print("\n----- ONNX Runtime Benchmark Results -----")
    print(f"Iterations:         {NUM_ITERS}")
    print(f"Average Latency:    {avg_lat:.2f} ms")
    print(f"P95 Latency:        {p95_lat:.2f} ms")
    print(f"Throughput:         {fps:.1f} FPS")
    print("------------------------------------------")
    
    # Inspect Output Shapes
    scores_out, boxes_out = raw_outs[0], raw_outs[1]
    print(f"Output 0 shape: {scores_out.shape} (scores/logits range: [{scores_out.min():.3f}, {scores_out.max():.3f}])")
    print(f"Output 1 shape: {boxes_out.shape} (boxes range: [{boxes_out.min():.3f}, {boxes_out.max():.3f}])")

In [ ]:
# CELL 6: Generate Production FastAPI Microservice & Container Files

# 1. Write requirements.txt
reqs_content = """fastapi>=0.110.0
uvicorn[standard]>=0.28.0
onnxruntime>=1.17.0
pillow>=10.2.0
numpy>=1.24.0
python-multipart>=0.0.9
pydantic>=2.0.0
"""
with open(ARTIFACTS_DIR / "requirements.txt", "w") as f:
    f.write(reqs_content)
print(f"✓ Generated {ARTIFACTS_DIR / 'requirements.txt'}")

# 2. Write app.py (FastAPI Server)
app_content = f"""# ==============================================================================
# RF-DETR ONNX Inference Microservice (FastAPI + ONNX Runtime)
# ==============================================================================
import os
import io
import time
from typing import List, Dict, Any, Optional

import numpy as np
from PIL import Image
from pydantic import BaseModel, Field
from fastapi import FastAPI, File, UploadFile, Query, HTTPException, status
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
import onnxruntime as ort

MODEL_PATH = os.getenv("MODEL_PATH", "rfdetr_model.onnx")
RESOLUTION = int(os.getenv("RESOLUTION", "{RESOLUTION}"))
CLASSES = {json.dumps(CLASSES)}

# Initialize FastAPI App
app = FastAPI(
    title="RF-DETR Object Detection API",
    description="Scalable ONNX Runtime Inference Microservice for RF-DETR on AKS",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Global Session
session = None
input_name = None
output_names = []

@app.on_event("startup")
def load_onnx_model():
    global session, input_name, output_names
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f"ONNX model file not found at: {{MODEL_PATH}}")
        
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if ort.get_device() == "GPU" else ["CPUExecutionProvider"]
    opts = ort.SessionOptions()
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    opts.intra_op_num_threads = int(os.getenv("ORT_NUM_THREADS", str(os.cpu_count() or 4)))
    
    session = ort.InferenceSession(MODEL_PATH, sess_options=opts, providers=providers)
    input_name = session.get_inputs()[0].name
    output_names = [o.name for o in session.get_outputs()]
    print(f"[STARTUP] Loaded ONNX model '{{MODEL_PATH}}' on {{session.get_providers()[0]}}")

# Response Schemas
class Detection(BaseModel):
    class_name: str
    class_id: int
    confidence: float
    box_xyxy: List[float] = Field(..., description="[x1, y1, x2, y2] in original image pixel coordinates")

class PredictionResponse(BaseModel):
    success: bool
    image_width: int
    image_height: int
    inference_time_ms: float
    detections_count: int
    detections: List[Detection]

@app.get("/healthz", status_code=status.HTTP_200_OK, tags=["Monitoring"])
def health_check():
    # Kubernetes Liveness and Readiness Probe Endpoint
    if session is None:
        raise HTTPException(status_code=503, detail="Model session not ready")
    return {{"status": "healthy", "model_loaded": True, "provider": session.get_providers()[0]}}

@app.get("/metadata", tags=["Metadata"])
def get_metadata():
    # Returns model architecture, resolution, and class catalogue
    return {{
        "model_path": MODEL_PATH,
        "resolution": RESOLUTION,
        "classes": CLASSES,
        "num_classes": len(CLASSES),
        "execution_provider": session.get_providers()[0] if session else "uninitialized"
    }}

def preprocess_image(image_bytes: bytes) -> Tuple[np.ndarray, int, int]:
    # Loads, resizes, and normalizes image for RF-DETR model
    pil_img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    orig_w, orig_h = pil_img.size
    
    resized = pil_img.resize((RESOLUTION, RESOLUTION), Image.BILINEAR)
    img_np = np.array(resized, dtype=np.float32) / 255.0
    
    # Standard ImageNet normalization: (x - mean) / std
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img_norm = (img_np - mean) / std
    
    # HWC to CHW -> add batch dimension (1, 3, H, W)
    tensor = np.transpose(img_norm, (2, 0, 1))[np.newaxis, ...]
    return tensor, orig_w, orig_h

@app.post("/predict", response_model=PredictionResponse, tags=["Inference"])
async def predict(
    file: UploadFile = File(..., description="JPEG/PNG image file"),
    conf_threshold: float = Query({CONF_THRESHOLD}, ge=0.0, le=1.0, description="Minimum detection score")
):
    # Runs detection on uploaded image and returns bounding boxes
    if session is None:
        raise HTTPException(status_code=503, detail="Model is still initializing")
        
    try:
        contents = await file.read()
        tensor, orig_w, orig_h = preprocess_image(contents)
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Failed to decode image: {{e}}")
        
    t_start = time.perf_counter()
    raw_outs = session.run(output_names, {{input_name: tensor}})
    inf_time_ms = round((time.perf_counter() - t_start) * 1000, 2)
    
    # Parse outputs
    out_scores = raw_outs[0][0]  # shape: (queries, num_classes)
    out_boxes  = raw_outs[1][0]  # shape: (queries, 4) in cx, cy, w, h
    
    # Apply sigmoid if scores are raw logits
    if out_scores.min() < 0.0 or out_scores.max() > 1.0:
        out_scores = 1.0 / (1.0 + np.exp(-out_scores))
        
    max_scores = np.max(out_scores, axis=-1)
    class_indices = np.argmax(out_scores, axis=-1)
    
    keep_mask = max_scores >= conf_threshold
    filtered_scores = max_scores[keep_mask]
    filtered_classes = class_indices[keep_mask]
    filtered_boxes = out_boxes[keep_mask]
    
    detections: List[Detection] = []
    for sc, cid, (cx, cy, w, h) in zip(filtered_scores, filtered_classes, filtered_boxes):
        # Convert cx, cy, w, h (normalized 0..1) to xyxy in original pixel dimensions
        x1 = float(max(0.0, (cx - w / 2.0) * orig_w))
        y1 = float(max(0.0, (cy - h / 2.0) * orig_h))
        x2 = float(min(float(orig_w), (cx + w / 2.0) * orig_w))
        y2 = float(min(float(orig_h), (cy + h / 2.0) * orig_h))
        
        c_name = CLASSES[cid] if cid < len(CLASSES) else f"class_{{cid}}"
        detections.append(Detection(
            class_name=c_name,
            class_id=int(cid),
            confidence=round(float(sc), 4),
            box_xyxy=[round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)]
        ))
        
    return PredictionResponse(
        success=True,
        image_width=orig_w,
        image_height=orig_h,
        inference_time_ms=inf_time_ms,
        detections_count=len(detections),
        detections=detections
    )
"""
with open(ARTIFACTS_DIR / "app.py", "w") as f:
    f.write(app_content)
print(f"✓ Generated {ARTIFACTS_DIR / 'app.py'}")

# 3. Write Dockerfile
dockerfile_content = """FROM python:3.10-slim

# Prevent Python from writing .pyc and buffer stdin/stdout
ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1
ENV MODEL_PATH="rfdetr_model.onnx"

WORKDIR /app

# Install system dependencies (curl for health check, libgomp1 for openmp)
RUN apt-get update && apt-get install -y --no-install-recommends \
    curl \
    libgomp1 \
    && rm -rf /var/lib/apt/lists/*

# Install python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy model and application code
COPY rfdetr_model.onnx .
COPY app.py .

# Create non-root user for security
RUN useradd -u 1000 appuser && chown -R appuser:appuser /app
USER appuser

EXPOSE 8000

# Health check instruction for container runtime
HEALTHCHECK --interval=30s --timeout=5s --start-period=10s --retries=3 \
    CMD curl -f http://localhost:8000/healthz || exit 1

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""
with open(ARTIFACTS_DIR / "Dockerfile", "w") as f:
    f.write(dockerfile_content)
print(f"✓ Generated {ARTIFACTS_DIR / 'Dockerfile'}")

In [ ]:
# CELL 7: In-Memory Verification of FastAPI Microservice
# Tests app endpoints locally using FastAPI TestClient to catch any bugs before container build.

from fastapi.testclient import TestClient

# Point environment to our exported ONNX model
os.environ["MODEL_PATH"] = str(ONNX_MODEL_PATH.resolve())

try:
    import importlib.util
    spec = importlib.util.spec_from_file_location("inference_app", str(ARTIFACTS_DIR / "app.py"))
    app_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(app_module)
    
    # Initialize the startup event
    app_module.load_onnx_model()
    client = TestClient(app_module.app)
    
    # 1. Test /healthz
    res_health = client.get("/healthz")
    print(f"Health Check Status: {res_health.status_code} -> {res_health.json()}")
    assert res_health.status_code == 200, "Health check failed!"
    
    # 2. Test /metadata
    res_meta = client.get("/metadata")
    print(f"Metadata Status:     {res_meta.status_code} -> {res_meta.json()}")
    assert res_meta.status_code == 200, "Metadata fetch failed!"
    
    # 3. Test /predict with dummy image
    img = Image.new("RGB", (640, 480), color=(128, 128, 128))
    img_byte_arr = io.BytesIO()
    img.save(img_byte_arr, format='JPEG')
    img_bytes = img_byte_arr.getvalue()
    
    res_pred = client.post(
        "/predict?conf_threshold=0.1",
        files={"file": ("test.jpg", img_bytes, "image/jpeg")}
    )
    print(f"Predict Status:      {res_pred.status_code}")
    print(f"Predict JSON:        {res_pred.json()}")
    assert res_pred.status_code == 200, "Predict endpoint failed!"
    
    print("\n=======================================================")
    print("✓ LOCAL MICROSERVICE VERIFICATION PASSED COMPLETELY")
    print("=======================================================")
except Exception as e:
    print(f"Notice during in-memory microservice test: {e}")

In [ ]:
# CELL 8: Generate Kubernetes Deployment, Service & HPA Manifests

# 1. deployment.yaml
k8s_deployment = """apiVersion: apps/v1
kind: Deployment
metadata:
  name: rfdetr-inference-deployment
  labels:
    app: rfdetr-inference
spec:
  replicas: 2
  selector:
    matchLabels:
      app: rfdetr-inference
  template:
    metadata:
      labels:
        app: rfdetr-inference
    spec:
      containers:
      - name: rfdetr-inference
        image: {{ACR_LOGIN_SERVER}}/rfdetr-inference:{{IMAGE_TAG}}
        imagePullPolicy: Always
        ports:
        - containerPort: 8000
          name: http
        resources:
          requests:
            cpu: "1000m"
            memory: "2Gi"
          limits:
            cpu: "2000m"
            memory: "4Gi"
        env:
        - name: MODEL_PATH
          value: "rfdetr_model.onnx"
        - name: RESOLUTION
          value: "{RESOLUTION}"
        - name: ORT_NUM_THREADS
          value: "2"
        livenessProbe:
          httpGet:
            path: /healthz
            port: 8000
          initialDelaySeconds: 20
          periodSeconds: 15
          timeoutSeconds: 5
        readinessProbe:
          httpGet:
            path: /healthz
            port: 8000
          initialDelaySeconds: 15
          periodSeconds: 10
          timeoutSeconds: 5
"""
with open(K8S_DIR / "deployment.yaml", "w") as f:
    f.write(k8s_deployment)
print(f"✓ Generated {K8S_DIR / 'deployment.yaml'}")

# 2. service.yaml (Azure Internal LoadBalancer - NO Public IP)
k8s_service = """apiVersion: v1
kind: Service
metadata:
  name: rfdetr-inference-service
  labels:
    app: rfdetr-inference
  annotations:
    service.beta.kubernetes.io/azure-load-balancer-internal: "true"
spec:
  type: LoadBalancer
  ports:
  - port: 80
    targetPort: 8000
    protocol: TCP
    name: http
  selector:
    app: rfdetr-inference
"""
with open(K8S_DIR / "service.yaml", "w") as f:
    f.write(k8s_service)
print(f"✓ Generated {K8S_DIR / 'service.yaml'} (Configured with internal load balancer annotation)")

# 3. hpa.yaml (Horizontal Pod Autoscaler)
k8s_hpa = """apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: rfdetr-inference-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: rfdetr-inference-deployment
  minReplicas: 2
  maxReplicas: 8
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        type: Utilization
        averageUtilization: 70
"""
with open(K8S_DIR / "hpa.yaml", "w") as f:
    f.write(k8s_hpa)
print(f"✓ Generated {K8S_DIR / 'hpa.yaml'}")

In [ ]:
# CELL 9: Azure Authentication & Resource Group Setup

def run_az_cmd(args: List[str], check: bool = True) -> Tuple[int, str]:
    """Helper to execute Azure CLI commands and stream output."""
    cmd = ["az"] + args
    print(f"Executing: {' '.join(cmd)}")
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if proc.returncode != 0 and check:
        print(f"Error output:\n{proc.stdout}")
        raise RuntimeError(f"Command failed with exit code {proc.returncode}")
    return proc.returncode, proc.stdout.strip()

# Check Azure Login Status
ret, out = run_az_cmd(["account", "show"], check=False)
if ret != 0:
    print("\n[ACTION REQUIRED]: You are not currently logged in to Azure CLI.")
    print("Please run:")
    print("   !az login")
    print("or log in using a Service Principal:")
    print("   !az login --service-principal -u <app-id> -p <password-or-cert> --tenant <tenant-id>")
else:
    account_info = json.loads(out)
    print(f"✓ Active Azure Account: {account_info.get('name')} (Subscription: {account_info.get('id')})")
    
    # Set subscription if user specified one
    if AZURE_SUBSCRIPTION_ID:
        run_az_cmd(["account", "set", "--subscription", AZURE_SUBSCRIPTION_ID])
        print(f"✓ Subscription set to: {AZURE_SUBSCRIPTION_ID}")
        
    # Create or update Resource Group
    print(f"Creating / ensuring Resource Group '{AZURE_RESOURCE_GROUP}' in region '{AZURE_LOCATION}'...")
    run_az_cmd(["group", "create", "--name", AZURE_RESOURCE_GROUP, "--location", AZURE_LOCATION])
    print(f"✓ Resource Group '{AZURE_RESOURCE_GROUP}' ready.")

In [ ]:
# CELL 10: Create Azure Container Registry (ACR) & Build Image via ACR Build
# Uses `az acr build`, which builds the Docker image directly inside Azure ACR.
# This does NOT require local Docker daemon to be running!

print(f"Creating Azure Container Registry '{ACR_NAME}' (SKU: Standard)...")
run_az_cmd([
    "acr", "create",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--name", ACR_NAME,
    "--sku", "Standard",
    "--location", AZURE_LOCATION
])

# Fetch ACR login server
_, out_login = run_az_cmd(["acr", "show", "--name", ACR_NAME, "--query", "loginServer", "-o", "tsv"])
ACR_LOGIN_SERVER = out_login.strip()
print(f"✓ ACR Login Server: {ACR_LOGIN_SERVER}")

# Cloud Container Build: packages ARTIFACTS_DIR and compiles image in ACR
FULL_IMAGE_TAG = f"{ACR_LOGIN_SERVER}/{IMAGE_NAME}:{IMAGE_TAG}"
print(f"\nSubmitting Cloud Container Build to ACR for '{FULL_IMAGE_TAG}'...")
print(f"Context directory: {ARTIFACTS_DIR.resolve()}")

run_az_cmd([
    "acr", "build",
    "--registry", ACR_NAME,
    "--image", f"{IMAGE_NAME}:{IMAGE_TAG}",
    str(ARTIFACTS_DIR)
])

print(f"\n✓ Container image successfully built and stored in ACR:")
print(f"   {FULL_IMAGE_TAG}")

In [ ]:
# CELL 11: Provision Azure Kubernetes Service (AKS) Cluster with ACR Integration

print(f"Provisioning AKS Cluster '{AKS_CLUSTER_NAME}'...")
print(f" - VM Size:        {NODE_VM_SIZE}")
print(f" - Node Count:     {NODE_COUNT}")
print(f" - Attached ACR:   {ACR_NAME} (Enables automatic kubelet image pull permissions)")

# Create AKS Cluster (or verify existing)
aks_cmd = [
    "aks", "create",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--name", AKS_CLUSTER_NAME,
    "--node-count", str(NODE_COUNT),
    "--node-vm-size", NODE_VM_SIZE,
    "--attach-acr", ACR_NAME,
    "--enable-managed-identity",
    "--generate-ssh-keys"
]

run_az_cmd(aks_cmd)
print(f"✓ AKS Cluster '{AKS_CLUSTER_NAME}' is ready.")

# Connect kubectl to the AKS cluster
print("Configuring kubectl context with cluster credentials...")
run_az_cmd([
    "aks", "get-credentials",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--name", AKS_CLUSTER_NAME,
    "--overwrite-existing"
])

# Verify kubectl connection
nodes_out = subprocess.check_output(["kubectl", "get", "nodes"], text=True)
print("\nKubernetes Cluster Nodes:")
print(nodes_out)

In [ ]:
# CELL 12: Deploy Manifests to AKS and Retrieve Private Internal Endpoint

# Substitute {{ACR_LOGIN_SERVER}} and {{IMAGE_TAG}} in deployment.yaml
with open(K8S_DIR / "deployment.yaml", "r") as f:
    template = f.read()

rendered_deployment = template.replace("{{ACR_LOGIN_SERVER}}", ACR_LOGIN_SERVER)
rendered_deployment = rendered_deployment.replace("{{IMAGE_TAG}}", IMAGE_TAG)

rendered_deployment_path = K8S_DIR / "deployment_rendered.yaml"
with open(rendered_deployment_path, "w") as f:
    f.write(rendered_deployment)

print(f"Applying Kubernetes manifests from {K8S_DIR}...")
subprocess.run(["kubectl", "apply", "-f", str(rendered_deployment_path)], check=True)
subprocess.run(["kubectl", "apply", "-f", str(K8S_DIR / "service.yaml")], check=True)
subprocess.run(["kubectl", "apply", "-f", str(K8S_DIR / "hpa.yaml")], check=True)

# Monitor deployment rollout status
print("\nWaiting for deployment rollout to complete...")
subprocess.run([
    "kubectl", "rollout", "status",
    "deployment/rfdetr-inference-deployment",
    "--timeout=300s"
], check=True)

print("✓ Pods are running successfully!")

# Poll for the Internal LoadBalancer Private IP (No Public IP)
print("\nPolling for Internal LoadBalancer Private IP (VNet-only address)...")
PRIVATE_IP = None
for attempt in range(24): # Poll up to 2 minutes
    out = subprocess.check_output([
        "kubectl", "get", "service", "rfdetr-inference-service",
        "-o", "jsonpath={.status.loadBalancer.ingress[0].ip}"
    ], text=True).strip()
    
    if out and out != "<pending>":
        PRIVATE_IP = out
        break
    time.sleep(5)
    print(f"Waiting for Private IP... (Attempt {attempt+1}/24)")

if PRIVATE_IP:
    print(f"\n=======================================================")
    print(f"✓ PRIVATE INTERNAL ENDPOINT READY (NO PUBLIC IP):")
    print(f"   - Private VNet IP: {PRIVATE_IP}")
    print(f"   - Internal URL:    http://{PRIVATE_IP}")
    print(f"   - Public Exposure: NONE (Azure Internal Load Balancer)")
    print(f"=======================================================\n")
else:
    print("⚠ Private IP assignment is still pending. Check status with: !kubectl get service rfdetr-inference-service")

# Setup kubectl port-forward tunnel for secure local/testing access
print("Setting up secure local port-forward tunnel (http://127.0.0.1:8000)...")
try:
    # Kill any stale port-forward on port 8000
    subprocess.run(["pkill", "-f", "kubectl port-forward service/rfdetr-inference-service"], stderr=subprocess.DEVNULL)
except Exception:
    pass

pf_proc = subprocess.Popen(
    ["kubectl", "port-forward", "service/rfdetr-inference-service", "8000:80"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(3)
if pf_proc.poll() is None:
    print("✓ Secure TLS Tunnel established: http://127.0.0.1:8000 (No Public IP required)")
    TARGET_ENDPOINT = "http://127.0.0.1:8000"
else:
    print(f"Notice: Port-forward process exited. Using internal IP directly: http://{PRIVATE_IP}")
    TARGET_ENDPOINT = f"http://{PRIVATE_IP}" if PRIVATE_IP else "http://127.0.0.1:8000"

In [ ]:
# CELL 13: Live Endpoint Verification via Private IP / Port-Forward & Visualization

import requests

API_URL = globals().get("TARGET_ENDPOINT", "http://127.0.0.1:8000")
print(f"Targeting Private Service Endpoint: {API_URL}")

# 1. Query Health Endpoint
try:
    res_health = requests.get(f"{API_URL}/healthz", timeout=10)
    print(f"Health Check: {res_health.status_code} -> {res_health.json()}")
except Exception as e:
    print(f"Could not connect to {API_URL}: {e}")
    if PRIVATE_IP:
        print(f"If running outside Azure VNet, make sure kubectl port-forward is running on port 8000.")

# 2. Select a test image from workspace or generate one
test_img_path = None
for cand_pattern in ["test_*.jpg", "*.jpg", "*.png", "coco_files/**/*.jpg"]:
    matches = list(Path(".").glob(cand_pattern))
    if matches:
        test_img_path = matches[0]
        break
        
if test_img_path and test_img_path.exists():
    print(f"Using local test image: {test_img_path}")
    with open(test_img_path, "rb") as f:
        img_bytes = f.read()
else:
    print("Creating synthetic sample image for endpoint verification...")
    test_img = Image.new("RGB", (800, 600), color=(230, 230, 230))
    d = ImageDraw.Draw(test_img)
    # Draw some colored synthetic shapes
    d.rectangle([100, 100, 300, 400], fill=(70, 130, 180), outline=(0, 0, 0), width=3)
    d.text((110, 110), "Test Object A", fill=(255, 255, 255))
    buf = io.BytesIO()
    test_img.save(buf, format="JPEG")
    img_bytes = buf.getvalue()

# 3. Send Inference Request
print(f"Sending POST request to {API_URL}/predict ...")
try:
    res_pred = requests.post(
        f"{API_URL}/predict",
        params={"conf_threshold": 0.15},
        files={"file": ("input.jpg", img_bytes, "image/jpeg")},
        timeout=30
    )
    
    print(f"Status Code: {res_pred.status_code}")
    pred_data = res_pred.json()
    print(f"Response Summary: {pred_data.get('detections_count', 0)} detections found in {pred_data.get('inference_time_ms', 0)} ms")

    # 4. Visualize Bounding Boxes
    pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(pil_img)
    
    colors = ["#ff3b30", "#34c759", "#007aff", "#5856d6", "#af52de", "#ff9500"]
    for det in pred_data.get("detections", []):
        x1, y1, x2, y2 = det["box_xyxy"]
        score = det["confidence"]
        cname = det["class_name"]
        cid = det["class_id"]
        c_color = colors[cid % len(colors)]
        
        # Draw rectangle
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=c_color, linewidth=2.5)
        ax.add_patch(rect)
        ax.text(
            x1, max(0, y1 - 8),
            f"{cname} ({score:.2f})",
            color="white",
            fontsize=10,
            fontweight="bold",
            bbox=dict(facecolor=c_color, alpha=0.8, edgecolor="none", pad=2)
        )
        
    ax.axis("off")
    plt.title(f"AKS Private Service Predictions ({pred_data.get('detections_count', 0)} tags detected)", fontsize=13, pad=10)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Prediction test notice: {e}")

In [ ]:
# CELL 14: Production Operations, Live Logs & Cleanup Utilities

# View live pod logs
print("=== Latest Pod Logs from AKS ===")
try:
    logs_out = subprocess.check_output([
        "kubectl", "logs",
        "-l", "app=rfdetr-inference",
        "--tail=30"
    ], text=True)
    print(logs_out)
except Exception as e:
    print(f"Notice: {e}")

# Check Horizontal Pod Autoscaler status
print("\n=== Autoscaler Status ===")
try:
    hpa_out = subprocess.check_output(["kubectl", "get", "hpa"], text=True)
    print(hpa_out)
except Exception as e:
    print(f"Notice: {e}")

# Manual Scaling Example
# Run: !kubectl scale deployment/rfdetr-inference-deployment --replicas=4

# ----------------- Resource Teardown -----------------
# When you have completed evaluation, uncomment and run the following command
# to delete the entire Azure Resource Group and stop all billing:

# print(f"Deleting Azure Resource Group '{AZURE_RESOURCE_GROUP}'...")
# !az group delete --name {AZURE_RESOURCE_GROUP} --yes --no-wait